# Индексация 50k постов в LanceDB (универсальный Colab-ноутбук)

Один параметризованный ноутбук для построения любой 50k-базы поиска:
`rosberta-base`, `e5-base-base`, `e5-base-fine-tuned`, `bge-m3-base`, `bge-m3-fine-tuned`.

## Что сделать перед запуском

**1. GPU-рантайм:** Runtime → Change runtime type → T4 GPU (или лучше).

**2. Данные на Google Drive** должны лежать так:

```
Google Drive/
└── thesis/
    ├── data/
    │   └── posts/ajtkulov/selected/selected500k_cleaned.jsonl
    └── models/   (если индексируешь fine-tuned модель)
        ├── bi-encoder-e5-finetuned.tar.gz       ← архив, как его сохраняет тренер
        └── bi-encoder-bge-m3-finetuned.tar.gz   ← архив, как его сохраняет тренер
```

Архивы fine-tuned моделей ноутбук распаковывает сам в `/content/` на Colab-VM — это быстрее, чем читать веса прямо с Drive.

**3. В секции CONFIG ниже раскомментируй ровно один пресет.** Между прогонами разных пресетов обязательно делай Runtime → Restart runtime, чтобы освободить VRAM.

**4. Локальная версия LanceDB** — впиши в `LANCEDB_VERSION` свою локальную версию (проверь: `python -c "import lancedb; print(lancedb.__version__)"`), иначе архив не откроется у тебя дома.

Результат: на Drive в `thesis/data/lancedb/` появится `<TABLE_NAME>_lance.tar.gz`. Скачиваешь, распаковываешь в `thesis/lancedb_store/`, готово.


In [ ]:
# CUDA-чек ДО установки пакетов (опционально, помогает ловить wrong runtime)
import torch
assert torch.cuda.is_available(), (
    "CUDA недоступна до установки пакетов. Значит, не выбран GPU-рантайм. "
    "Runtime → Change runtime type → GPU."
)
_initial_torch = torch.__version__
print(f"Перед install: torch={_initial_torch}, CUDA={torch.cuda.is_available()}")


In [ ]:
# Установка зависимостей и монтирование Drive
LANCEDB_VERSION = "0.21.1"   # <-- поставь свою локальную версию

# --upgrade-strategy only-if-needed, чтобы pip НЕ переустановил
# родной колабовский CUDA-torch на CPU-only версию из зависимостей.
!pip install -q --upgrade-strategy only-if-needed \
    lancedb=={LANCEDB_VERSION} tantivy sentence-transformers pyarrow tqdm

from google.colab import drive
drive.mount('/content/drive')

# CUDA-чек ПОСЛЕ install — убедиться, что pip не сломал torch
import importlib, torch as _torch_check
importlib.reload(_torch_check)
assert _torch_check.cuda.is_available(), (
    "CUDA сломалась после pip install. Скорее всего, pip переустановил torch "
    "на CPU-only версию. Решение: Runtime → Restart runtime и запусти ячейки заново."
)
print(f"После install: torch={_torch_check.__version__}, GPU={_torch_check.cuda.get_device_name(0)}")


## Конфигурация модели

Раскомментируй **один** из пресетов ниже.


In [ ]:
# ==================== CONFIG: выбери ОДНУ модель ====================
# MODEL_SOURCE может быть:
#   - HF-идентификатор ("intfloat/multilingual-e5-base")
#   - путь к .tar.gz на Drive (распакуется в /content/ автоматически)
#   - путь к уже распакованной папке на Drive или локально

# --- RoSBERTa base (для сравнения с твоим дообученным вариантом) ---
# MODEL_SOURCE  = "ai-forever/ru-en-RoSBERTa"
# TABLE_NAME    = "rosberta-base-50k"
# DOC_PREFIX    = ""
# QUERY_PREFIX  = ""
# BATCH_SIZE    = 384

# --- E5-base zero-shot (уже есть локально — пересоздать при необходимости) ---
# MODEL_SOURCE  = "intfloat/multilingual-e5-base"
# TABLE_NAME    = "e5-base-base-50k"
# DOC_PREFIX    = "passage: "
# QUERY_PREFIX  = "query: "
# BATCH_SIZE    = 384

# --- E5-base fine-tuned (.tar.gz с Drive, распакуется в /content/) ---
MODEL_SOURCE  = "/content/drive/MyDrive/thesis/models/bi-encoder-e5-finetuned.tar.gz"
TABLE_NAME    = "e5-base-fine-tuned-50k"
DOC_PREFIX    = "passage: "
QUERY_PREFIX  = "query: "
BATCH_SIZE    = 384

# --- USER-bge-m3 base (с HuggingFace) ---
# MODEL_SOURCE  = "deepvk/USER-bge-m3"
# TABLE_NAME    = "bge-m3-base-50k"
# DOC_PREFIX    = ""
# QUERY_PREFIX  = ""
# BATCH_SIZE    = 128    # bge-m3 крупнее (568M params), 384 на T4 не влезет

# --- USER-bge-m3 fine-tuned (.tar.gz с Drive, после твоего обучения) ---
# MODEL_SOURCE  = "/content/drive/MyDrive/thesis/models/bi-encoder-bge-m3-finetuned.tar.gz"
# TABLE_NAME    = "bge-m3-fine-tuned-50k"
# DOC_PREFIX    = ""
# QUERY_PREFIX  = ""
# BATCH_SIZE    = 128

# --------- общие параметры (одинаковые для всех моделей) ---------
MAX_POSTS      = 50_000
MAX_SEQ_LEN    = 256       # обрезаем длинные посты — ускоряет кодирование
RANDOM_SEED    = 42
DEVICE         = "cuda"
CLEAN_CATEGORY = True      # оставить только первую категорию до '|||'

print(f"Пресет: {TABLE_NAME}")
print(f"MODEL_SOURCE: {MODEL_SOURCE}")
print(f"DOC_PREFIX: '{DOC_PREFIX}'  QUERY_PREFIX: '{QUERY_PREFIX}'")
print(f"BATCH_SIZE={BATCH_SIZE}  MAX_POSTS={MAX_POSTS}  MAX_SEQ_LEN={MAX_SEQ_LEN}")


In [ ]:
# Подготовка MODEL_PATH: если source — .tar.gz с Drive, распаковываем в /content/.
# Это быстрее, чем грузить веса прямо с Drive (там мелкие файлы читаются очень долго).
import os, tarfile, time

LOCAL_MODELS_DIR = "/content/models"
os.makedirs(LOCAL_MODELS_DIR, exist_ok=True)

def resolve_model(source: str) -> str:
    # HF-идентификатор (org/model) — отдаём как есть, SentenceTransformer сам скачает
    if not source.startswith("/") and "/" in source and not source.endswith(".tar.gz"):
        return source

    # Архив на Drive / локально — распаковать в /content/models/<name>
    if source.endswith(".tar.gz"):
        assert os.path.exists(source), f"Архив модели не найден: {source}"
        base_name = os.path.basename(source)[:-len(".tar.gz")]
        extracted = os.path.join(LOCAL_MODELS_DIR, base_name)
        if os.path.exists(extracted) and os.listdir(extracted):
            print(f"Модель уже распакована: {extracted}")
            return extracted
        print(f"Распаковка {source} → {LOCAL_MODELS_DIR} ...")
        t0 = time.time()
        with tarfile.open(source, "r:gz") as tar:
            tar.extractall(LOCAL_MODELS_DIR)
        print(f"  готово за {time.time()-t0:.0f}с")
        # tar создаёт папку с именем base_name (см. тренировочный ноутбук)
        assert os.path.isdir(extracted), (
            f"После распаковки ожидалась папка {extracted}, но её нет. "
            f"Посмотри, что внутри {LOCAL_MODELS_DIR}: {os.listdir(LOCAL_MODELS_DIR)}"
        )
        return extracted

    # Обычный локальный путь к папке
    assert os.path.isdir(source), f"Папка с моделью не найдена: {source}"
    return source

MODEL_PATH = resolve_model(MODEL_SOURCE)
print(f"MODEL_PATH для SentenceTransformer: {MODEL_PATH}")


In [ ]:
# Импорты, пути, sanity-чеки
import json, random, shutil
from tqdm.auto import tqdm
import numpy as np
import pyarrow as pa
import lancedb
import warnings; warnings.filterwarnings('ignore')

DRIVE_BASE         = "/content/drive/MyDrive/thesis"
INPUT_JSONL        = os.path.join(DRIVE_BASE, "data/posts/ajtkulov/selected/selected500k_cleaned.jsonl")
DRIVE_OUTPUT_DIR   = os.path.join(DRIVE_BASE, "data/lancedb")
LOCAL_LANCEDB_PATH = "/content/lancedb_store"

os.makedirs(DRIVE_OUTPUT_DIR, exist_ok=True)

assert os.path.exists(INPUT_JSONL), f"Входной файл не найден: {INPUT_JSONL}"
print(f"Входной файл: {INPUT_JSONL}  ({os.path.getsize(INPUT_JSONL)/1e6:.1f} MB)")
print(f"Drive output:  {DRIVE_OUTPUT_DIR}")
print(f"Local LanceDB: {LOCAL_LANCEDB_PATH}")


In [ ]:
# Загрузка модели
from sentence_transformers import SentenceTransformer

bi_encoder = SentenceTransformer(MODEL_PATH, device=DEVICE)
bi_encoder.max_seq_length = MAX_SEQ_LEN
EMBEDDING_DIM = bi_encoder.get_sentence_embedding_dimension()
print(f"Модель загружена: {MODEL_PATH}")
print(f"  dim: {EMBEDDING_DIM}")
print(f"  max_seq_len: {bi_encoder.max_seq_length}")


In [ ]:
# Чтение и семплирование постов
all_posts = []
with open(INPUT_JSONL, 'r', encoding='utf-8') as f:
    for line in tqdm(f, desc="Чтение постов"):
        obj = json.loads(line)
        if not obj.get('text', '').strip():
            continue
        if CLEAN_CATEGORY:
            cat = obj.get('category', '')
            if '|||' in cat:
                obj['category'] = cat.split('|||')[0].strip()
        all_posts.append(obj)

print(f"Всего постов в файле: {len(all_posts):,}")

rng = random.Random(RANDOM_SEED)
posts = rng.sample(all_posts, MAX_POSTS)
print(f"Случайная выборка (seed={RANDOM_SEED}): {len(posts):,}")


In [ ]:
# Создание локальной LanceDB с корректной схемой под dim модели
db = lancedb.connect(LOCAL_LANCEDB_PATH)

schema = pa.schema([
    pa.field("vector",   pa.list_(pa.float32(), EMBEDDING_DIM)),
    pa.field("text",     pa.utf8()),
    pa.field("channel",  pa.utf8()),
    pa.field("category", pa.utf8()),
    pa.field("post_id",  pa.utf8()),
    pa.field("link",     pa.utf8()),
    pa.field("date",     pa.utf8()),
    pa.field("views",    pa.utf8()),
])

if TABLE_NAME in db.table_names():
    db.drop_table(TABLE_NAME)
    print(f"Старая таблица '{TABLE_NAME}' удалена")

table = db.create_table(TABLE_NAME, schema=schema)
print(f"Создана таблица '{TABLE_NAME}' в {LOCAL_LANCEDB_PATH} (dim={EMBEDDING_DIM})")


In [ ]:
# Векторизация и запись
def make_post_id(post):
    return f"{post['channel']}::{post['id']}"

total = len(posts)
print(f"Старт: {total:,} постов, batch={BATCH_SIZE}, device={DEVICE}")

t_start = time.time()
indexed = 0
pbar = tqdm(total=total, desc="Векторизация", unit="пост")

for batch_start in range(0, total, BATCH_SIZE):
    batch = posts[batch_start : batch_start + BATCH_SIZE]
    texts = [p['text'] for p in batch]
    encoded_texts = [DOC_PREFIX + t for t in texts] if DOC_PREFIX else texts

    embeddings = bi_encoder.encode(
        encoded_texts,
        normalize_embeddings=True,
        show_progress_bar=False,
        batch_size=BATCH_SIZE,
        device=DEVICE,
    )

    records = [{
        "vector":   embeddings[i].tolist(),
        "text":     batch[i]['text'],           # без префикса
        "channel":  batch[i]['channel'],
        "category": batch[i].get('category', ''),
        "post_id":  make_post_id(batch[i]),
        "link":     batch[i].get('link', ''),
        "date":     batch[i].get('date', ''),
        "views":    str(batch[i].get('views', '')),
    } for i in range(len(batch))]

    table.add(records)
    indexed += len(records)

    elapsed = time.time() - t_start
    speed = indexed / elapsed
    eta = (total - indexed) / speed if speed > 0 else 0
    pbar.update(len(records))
    pbar.set_postfix({"скорость": f"{speed:.0f} п/с", "ETA": f"{eta/60:.1f} мин"})

pbar.close()
print(f"\nГотово: {indexed:,} постов за {time.time()-t_start:.0f}с")
print(f"В таблице: {table.count_rows():,} записей")


In [ ]:
# FTS-индекс (BM25)
print("Создание FTS-индекса на поле 'text'...")
table.create_fts_index("text", replace=True)
print("Готово")


In [ ]:
# Санити-чек: тестовый запрос
test_query = "Кроссовки Nike Air Max мужские для бега, размер 42, чёрные"
print(f"Запрос: {test_query}\n")

q_for_enc = (QUERY_PREFIX + test_query) if QUERY_PREFIX else test_query
qvec = bi_encoder.encode([q_for_enc], normalize_embeddings=True)[0].tolist()

print("=== ВЕКТОРНЫЙ ПОИСК ===")
for i, r in enumerate(
    table.search(qvec, query_type="vector").limit(5).select(["text", "channel", "category"]).to_list(), 1
):
    print(f"  {i}. [{r['category']}] @{r['channel']}")
    print(f"     {r['text'][:120]}...")
    print()

print("=== BM25 ===")
for i, r in enumerate(
    table.search(test_query, query_type="fts").limit(5).select(["text", "channel", "category"]).to_list(), 1
):
    print(f"  {i}. [{r['category']}] @{r['channel']}")
    print(f"     {r['text'][:120]}...")
    print()


In [ ]:
# Статистика размера таблицы
def dir_size_mb(path):
    total = 0
    for dirpath, _, filenames in os.walk(path):
        for f in filenames:
            total += os.path.getsize(os.path.join(dirpath, f))
    return total / (1024 * 1024)

table_dir = os.path.join(LOCAL_LANCEDB_PATH, f"{TABLE_NAME}.lance")
rows = table.count_rows()
size = dir_size_mb(table_dir)

print(f"Таблица: {TABLE_NAME}")
print(f"Записей:  {rows:,}")
print(f"Размер:   {size:.1f} MB")
print(f"KB/запись: {size * 1024 / rows:.1f}")


In [ ]:
# Архивирование таблицы и копирование на Drive
ARCHIVE_NAME  = f"{TABLE_NAME}_lance.tar.gz"
local_archive = f"/content/{ARCHIVE_NAME}"

assert os.path.exists(table_dir), f"Таблица не найдена на диске: {table_dir}"

!tar -czf {local_archive} -C {LOCAL_LANCEDB_PATH} {TABLE_NAME}.lance

archive_size_mb = os.path.getsize(local_archive) / 1e6
print(f"Архив создан: {local_archive}  ({archive_size_mb:.1f} MB)")

drive_archive = os.path.join(DRIVE_OUTPUT_DIR, ARCHIVE_NAME)
shutil.copy(local_archive, drive_archive)
assert os.path.exists(drive_archive), "Не удалось скопировать архив на Drive"
print(f"✓ Архив сохранён на Drive: {drive_archive}")
print(f"  Размер: {os.path.getsize(drive_archive) / 1e6:.1f} MB")


## Что делать локально после скачивания

Забираешь с Drive файл `thesis/data/lancedb/<TABLE_NAME>_lance.tar.gz` и распаковываешь в `thesis/lancedb_store/`:

```bash
cd thesis/
tar -xzf /path/to/<TABLE_NAME>_lance.tar.gz -C lancedb_store/
```

Появится `thesis/lancedb_store/<TABLE_NAME>.lance/`. После этого бенчмарки и тестовые ноутбуки, у которых прописано это имя таблицы, будут её видеть.

Если локально уже есть таблица с тем же именем — удали её руками (`rm -rf thesis/lancedb_store/<TABLE_NAME>.lance`) и повтори распаковку.

## Прогоны по очереди

Чтобы построить несколько баз подряд:
1. Прогнал один пресет → архив ушёл на Drive.
2. Runtime → Restart runtime (обязательно — иначе VRAM от предыдущей модели занята).
3. Раскомментировал следующий пресет в CONFIG, закомментировал предыдущий.
4. Run all cells.
